In [174]:
# это хорошая пробная версия, но ещё нужны доработки
# я работаю над ней
import requests
import pandas as pd
from bs4 import BeautifulSoup

class HTMLTableParser:
    """
    This class is parse have two method.
    It parse HTML as a string and then convert this into Excel table.
    """
    
    def parser_url(self):
        """
        This method parse the HTML as a string and search all tables.
        """ 
        
        response = requests.get('https://dance.vftsarr.ru/reg_module/?mode=reglists&competition_id=97')
        soup = BeautifulSoup(response.content, 'lxml') 
        table = soup.find('table', class_ = 'table')
        return [table, self.parse_html_table(table)]
    
    def parse_html_table(self, table):
        """
        This method create a new table of data from HTML.
        """
        
        """
        First things first, find number of rows and columns. 
        Also, find the columns titles.
        """
        
        n_columns = len(table.find_all('th', scope = 'col')) - 1     # Determine the number of rows in the table.
        n_rows = len(table.find_all('th', scope = 'row'))            # Set the number of columns for table.
        column_names = []

        """
        Handle column names wnen find them.
        """
        
        th_tags = table.find_all('th', scope = 'col')
        if len(th_tags) > 0 and len(column_names) == 0:
            for th in th_tags:
                column_names.append(th.get_text())
        del column_names[0]
        """
        Save column titles.
        """
        
        if len(column_names) > 0 and len(column_names) != n_columns:
            raise Exception("Column titles do not match the number of columns")
            
        columns = column_names if len(column_names) > 0 else range(0, n_columns)
        df = pd.DataFrame(columns = columns, index = range(1, n_rows+1))
        df.index.names = [table.find('th', scope = 'col').text]
        row_marker = 0
        for row in table.find_all('tr'):
            column_marker = 0
            columns = row.find_all('td')
            for column in columns:
                df.iat[row_marker,column_marker] = column.get_text().strip()
                column_marker += 1
            if len(columns) > 0:
                row_marker += 1
                
        """
        Convert to float if possible.
        """
        
        for col in df:
            try:
                df[col] = df[col].astype(float)
            except ValueError:
                pass
            
        return df, df_p.to_excel('Участники всех категорий.xlxs')
    
    def category_information(self, table):
        """
        This method search all links in HTML and return information of category in table.
        """    
        
        link_url = []
        link_col_name = []
        n_link_col = 0
        n_rows = 0
        
        for link in table.find_all('a'):
            if link.has_attr('href'):
                link_url.append(link['href'])
                
        response = requests.get(link_url[0])
        soup = BeautifulSoup(response.content, 'lxml') 
        table_p = soup.find('table', class_ = 'table')
        
        th_tags = table_p.find_all('th', scope = 'col')
        n_link_col += len(th_tags)
        if len(th_tags) > 0 and len(link_col_name) == 0:
            for th in th_tags:
                link_col_name.append(th.get_text())
        
        if len(link_col_name) > 0 and len(link_col_name) != n_link_col:
            raise Exception("Column titles do not match the number of columns")
          
        for link in link_url:
            response = requests.get(link)
            soup = BeautifulSoup(response.content, 'lxml') 
            table_p = soup.find('table', class_ = 'table')
            n_rows += len(table_p.find_all('tr'))
            n_rows -= 1
        
        columns = link_col_name if len(link_col_name) > 0 else range(0, n_link_col)
        df_p = pd.DataFrame(columns = columns, index = range(1, (n_rows+1)))
        row_number = 0
        for link in link_url:
            response = requests.get(link)
            soup = BeautifulSoup(response.content, 'lxml') 
            table_p = soup.find('table', class_ = 'table')
            for row in table_p.find_all('tr'):
                column_number = 0
                columns = row.find_all('th', scope='row')
                for br in soup.find_all('br'):
                    br.replace_with("%s\n " % br.text)
                for column in columns:
                    df_p.iat[row_number,column_number] = column.get_text().strip()
                    column_number += 1
                if len(columns) > 0:
                    row_number += 1
                    
        df_p.to_excel('Участники всех категорий.xlxs')
        return df_p